# Medical Form Understanding & Population Pipeline

This notebook implements an AI-assisted document understanding pipeline for medical forms.

## Overview
1. **Schema Extraction** - Extract form field schemas from fillable PDFs
2. **Information Extraction** - Extract patient information from documents
3. **Field Mapping** - Map extracted data to form fields with confidence scores
4. **PDF Population** - Populate the form with extracted values
5. **Evaluation** - Test and validate the pipeline

---

## Task 1: Schema Extraction

Extract field schemas from the fillable PDF, including labels, types, bounding boxes, and normalized names.

In [1]:
# Import schema extraction module
import sys
sys.path.append('src')

from schema_extraction import SchemaExtractor, extract_schema

# Extract schema from fillable PDF
pdf_path = "form_fillable.pdf"
schema = extract_schema(pdf_path, output_path="schema.json", filter_clear_button=True)

print(f"✅ Extracted schema with {len(schema)} fields")
print("\nSample fields:")
for i, (key, field) in enumerate(list(schema.items())[:5]):
    print(f"\n{key}:")
    print(f"  Label: {field['label']}")
    print(f"  PDF Name: {field['pdf_name']}")
    print(f"  Type: {field['type']}")
    if field.get('group'):
        print(f"  Group: {field['group']}, Part: {field['part']}")


✅ Extracted schema with 49 fields

Sample fields:

home_phone_area_code:
  Label: Home Phone # (+ Area Code)
  PDF Name: APS1. areacode
  Type: text
  Group: home_phone, Part: area_code

address_street_city_province_postal_code:
  Label: Address (Street, City, Province, Postal Code)
  PDF Name: APS1.Address
  Type: text

certificate_if_applicable:
  Label: Certificate # (if applicable)
  PDF Name: APS1.Cert
  Type: text

contract_or_policy:
  Label: Contract or Policy #
  PDF Name: APS1.Contract
  Type: text

date_last_day:
  Label: Date Last Worked (dd)
  PDF Name: APS1.Date_last_d
  Type: text
  Group: date_last, Part: day


## Task 2: Information Extraction

Extract patient information from documents (demographics.json, soap_notes.txt, lab_result.pdf).

*Module to be created: `src/information_extraction.py`*


In [2]:
# Extract information from demographics.json using the module
import sys
sys.path.append('src')

from information_extraction import InformationExtractor
import json

# Load schema
with open('schema.json', 'r') as f:
    schema = json.load(f)

# Create extractor
extractor = InformationExtractor(schema)

# Extract from demographics.json
print("="*60)
print("EXTRACTING FROM DEMOGRAPHICS.JSON")
print("="*60)

results = extractor.extract_from_demographics('demographics.json', confidence=0.95)

print(f"\n✅ Extracted {len(results)} fields from demographics.json")
print("\n📊 Extracted Fields:")
for field_name, field_data in sorted(results.items()):
    print(f"\n  {field_name}:")
    print(f"    Value: {field_data['value']}")
    print(f"    Source: {field_data['source']}")
    print(f"    Confidence: {field_data['confidence']}")
    print(f"    Transformation: {field_data['transformation']}")

# Show phone parsing verification
print("\n" + "="*60)
print("PHONE PARSING VERIFICATION")
print("="*60)

phone_groups = {
    "home_phone": ["home_phone_area_code", "home_phone_first3", "home_phone_last4"],
    "cell_phone": ["cell_phone_area_code", "cell_phone_first3", "cell_phone_last4"]
}

for phone_type, field_list in phone_groups.items():
    print(f"\n{phone_type.upper().replace('_', ' ')}:")
    all_present = all(field in results for field in field_list)
    if all_present:
        area = results[field_list[0]]['value']
        first3 = results[field_list[1]]['value']
        last4 = results[field_list[2]]['value']
        print(f"  ✓ Area Code: {area}")
        print(f"  ✓ First 3: {first3}")
        print(f"  ✓ Last 4: {last4}")
        print(f"  → Combined: {area}-{first3}-{last4}")
    else:
        print(f"  ✗ Missing fields")


EXTRACTING FROM DEMOGRAPHICS.JSON

✅ Extracted 11 fields from demographics.json

📊 Extracted Fields:

  address_street_city_province_postal_code:
    Value: 45 Maple Ave, Toronto, ON, K7L 3V8
    Source: demographics.json
    Confidence: 0.95
    Transformation: address_format

  cell_phone_area_code:
    Value: 647
    Source: demographics.json
    Confidence: 0.95
    Transformation: phone_parse.area_code

  cell_phone_first3:
    Value: 666
    Source: demographics.json
    Confidence: 0.95
    Transformation: phone_parse.first3

  cell_phone_last4:
    Value: 8888
    Source: demographics.json
    Confidence: 0.95
    Transformation: phone_parse.last4

  date_of_birth_dd:
    Value: 15
    Source: demographics.json
    Confidence: 0.95
    Transformation: date_parse.day

  date_of_birth_mm:
    Value: 04
    Source: demographics.json
    Confidence: 0.95
    Transformation: date_parse.month

  date_of_birth_yyyy:
    Value: 1960
    Source: demographics.json
    Confidence: 0.95
  

In [3]:
# ============================================================================
# SOAP extraction (in src/), completion-only
# ============================================================================
# - Demographics is the anchor truth for identity/contact fields.
# - SOAP is used to fill missing clinical fields (diagnoses, explicit meds, etc.).
# - Deterministic-first, evidence-gated; optional LLM fallback for remaining missing keys.

# (Re-)load SOAP text for downstream evaluation cell
with open('soap_notes.txt', 'r') as f:
    soap_text = f.read()

soap_results = extractor.extract_from_soap_notes(
    'soap_notes.txt',
    already_filled=results,
    use_llm_fallback=True,
    model_name='Qwen/Qwen2.5-3B-Instruct',
    model_cache_dir='./models/Qwen2.5-3B-Instruct',
)

kept = {k: v for k, v in soap_results.items() if v.get('value') is not None}
print('=' * 60)
print('SOAP EXTRACTION (src/information_extraction.py)')
print('=' * 60)
print(f"✅ Kept fields: {len(kept)}")
for k, v in kept.items():
    print(f"\n{k}:")
    print(f"  value: {v['value']}")
    print(f"  confidence: {v['confidence']}")
    print(f"  evidence: {v['evidence']}")
    print(f"  reasoning: {v['reasoning']}")



The tokenizer you are loading from 'models/Qwen2.5-3B-Instruct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SOAP EXTRACTION (src/information_extraction.py)
✅ Kept fields: 6

primary_1:
  value: stable angina given exertional pattern and resolution w/ rest
  confidence: 0.75
  evidence: Likely stable angina given exertional pattern and resolution w/ rest. Multiple risk factors: HTN, age, family hx (father MI at 58).
  reasoning: extracted from Assessment section

primary_2:
  value: Hypertension
  confidence: 0.75
  evidence: Hypertension — borderline control; adherence inconsistent.
  reasoning: extracted from Assessment section

secondary_and_or_complications_1:
  value: GERD
  confidence: 0.75
  evidence: GERD — chronic but unlikely to explain exertional symptoms.
  reasoning: extracted from Assessment section

secondary_and_or_complications_2:
  value: Hyperlipidemia
  confidence: 0.75
  evidence: Hyperlipidemia — labs overdue.
  reasoning: extracted from Assessment section

medication_1_name:
  value: nitroglycerin
  confidence: 0.8
  evidence: Provide nitroglycerin SL for exertional dis

In [4]:
# ============================================================================
# Reconciliation: demographics (anchor) + SOAP (fill missing / flag conflicts)
# ============================================================================
# Assumptions:
# - `results` exists from the demographics extraction cell.
# - `soap_results` and `soap_text` exist from the SOAP extraction cell.

import re


def _norm(s):
    if s is None:
        return None
    return re.sub(r"\s+", " ", str(s)).strip().lower()

combined_results = dict(results)  # start with high-confidence structured values
conflicts = []
verified = []
added_from_soap = []

for k, soap_obj in soap_results.items():
    soap_val = soap_obj.get("value")
    if soap_val is None:
        continue

    if k not in combined_results:
        combined_results[k] = soap_obj
        added_from_soap.append(k)
        continue

    demo_val = combined_results[k].get("value")
    if _norm(demo_val) == _norm(soap_val):
        # Keep demographics, just record that SOAP corroborated it
        combined_results[k] = {
            **combined_results[k],
            "verified_by": "soap_notes.txt",
            "verification_evidence": soap_obj.get("evidence"),
        }
        verified.append(k)
    else:
        # Do not overwrite; flag for human review
        conflicts.append({
            "field": k,
            "demographics_value": demo_val,
            "soap_value": soap_val,
            "soap_evidence": soap_obj.get("evidence"),
        })
        combined_results[k] = {
            **combined_results[k],
            "needs_review": True,
            "conflict_with": "soap_notes.txt",
            "conflict_candidate": soap_val,
            "conflict_evidence": soap_obj.get("evidence"),
        }

print("=" * 60)
print("RECONCILIATION SUMMARY")
print("=" * 60)
print(f"Added from SOAP: {len(added_from_soap)} -> {added_from_soap}")
print(f"Verified by SOAP: {len(verified)} -> {verified}")
print(f"Conflicts flagged: {len(conflicts)}")

if conflicts:
    print("\n⚠️  Conflicts (kept demographics; marked needs_review):")
    for c in conflicts:
        print(f"\n- {c['field']}")
        print(f"  demographics: {c['demographics_value']}")
        print(f"  soap: {c['soap_value']}")
        print(f"  evidence: {c['soap_evidence']}")

print("\n✅ Combined extracted fields:", len(combined_results))

# ============================================================================
# Mini-eval (hallucination-focused)
# ============================================================================
# 1) Evidence match rate on SOAP-kept fields
# 2) Dose hallucination trap: dose fields should be null unless explicitly numeric in note

soap_kept = {k: v for k, v in soap_results.items() if v.get("value") is not None}
soap_evidence_ok = 0

soap_text_norm = re.sub(r"\s+", " ", soap_text).strip()
for k, v in soap_kept.items():
    ev = v.get("evidence")
    if isinstance(ev, str) and re.sub(r"\s+", " ", ev).strip() in soap_text_norm:
        soap_evidence_ok += 1

print("\n" + "=" * 60)
print("SOAP MINI-EVAL")
print("=" * 60)
print(f"SOAP kept fields: {len(soap_kept)}")
print(f"Evidence match rate (kept): {soap_evidence_ok}/{len(soap_kept)}" if soap_kept else "Evidence match rate (kept): n/a")

bad_doses = []
for k, v in soap_kept.items():
    if k.endswith("_dose"):
        if not re.search(r"\d", str(v.get("value"))):
            bad_doses.append(k)

print(f"Dose hallucination checks (should be empty): {bad_doses}")



RECONCILIATION SUMMARY
Added from SOAP: 6 -> ['primary_1', 'primary_2', 'secondary_and_or_complications_1', 'secondary_and_or_complications_2', 'medication_1_name', 'medication_1_often']
Verified by SOAP: 0 -> []
Conflicts flagged: 0

✅ Combined extracted fields: 17

SOAP MINI-EVAL
SOAP kept fields: 6
Evidence match rate (kept): 6/6
Dose hallucination checks (should be empty): []


In [ ]:
# ============================================================================
# Lab result extraction (generic, optional): fill only remaining missing fields
# ============================================================================
# Per problem.md: lab_result.pdf can contain anything, so we keep this conservative and generic.
# - Prefer embedded PDF text extraction; OCR is only used as a fallback if the PDF has no text.
# - Only fill fields that are still missing after demographics + SOAP.

lab_results = extractor.extract_from_lab_result(
    'lab_result.pdf',
    already_filled=combined_results,
    use_ocr_fallback=True,
)

added = sorted([k for k, v in lab_results.items() if v.get('value') is not None])
print('=' * 60)
print('LAB RESULT EXTRACTION')
print('=' * 60)
print(f"✅ Added fields from lab_result.pdf: {len(added)}")
print('   ', added)

# Merge into combined results
combined_results = extractor.combine_extractions(combined_results, lab_results)
print(f"✅ Combined extracted fields after lab_result.pdf: {len(combined_results)}")



## Task 4: PDF Population (Optional)

Populate the fillable PDF with extracted values.

*Module to be created: `src/pdf_population.py`*


## Task 5: Evaluation

Test and validate the pipeline with evaluation metrics.

*Module to be created: `src/evaluation.py`*
